In [8]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 

In [9]:
# tokens start from 1
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-64
      
            self.y[ii] = \
                ord(data[ii+jj+1])-64

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [23]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim+context_size, hidden_size, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, context, h=None):
        embedded = self.embedding(x)
        # print(embedded, context)
        embedded = torch.cat((embedded, context), axis=2)
        out, h = self.rnn(embedded, h)
        out = self.linear(out[:,-1,:])

        return out, h  

In [25]:
### initial training ###
total_samples = 30000
working_memory = 1
short_term_memory = 1
hidden_size = 50
vocab_size = 8
context_size=30
embedding_dim = 5
lr = 1e-3
test_acc = []
test_acc_decoder = []

model = RNNEncoder(vocab_size, embedding_dim, context_size, hidden_size)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95, weight_decay=1e-8)
criterion = torch.nn.CrossEntropyLoss()

data = get_sequence(total_samples, 2, 3, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

total = 0
correct = np.zeros(1000,dtype=float)
decoder_correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    context = torch.rand((1,1,context_size))

    optimizer.zero_grad()

    if total == 0:
        y_pred, h = model(X, context)
    else:
        y_pred, h = model(X, context, hidden)

    loss = criterion(y_pred, y[0])     
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        hidden = h.clone()
        total += 1

        if y[0] == y_pred[0].argmax():
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        
        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')
    
        

Iter : 1001, loss: 1.8362, accuracy: 0.3550
Iter : 2001, loss: 1.4932, accuracy: 0.5000
Iter : 3001, loss: 1.9984, accuracy: 0.6690
Iter : 4001, loss: 1.8533, accuracy: 0.6770
Iter : 5001, loss: 2.1182, accuracy: 0.6520
Iter : 6001, loss: 1.5884, accuracy: 0.6660
Iter : 7001, loss: 1.9807, accuracy: 0.6590
Iter : 8001, loss: 1.8090, accuracy: 0.6790
Iter : 9001, loss: 2.0941, accuracy: 0.6620
Iter : 10001, loss: 1.7318, accuracy: 0.6630
Iter : 11001, loss: 1.8822, accuracy: 0.6470
Iter : 12001, loss: 1.8063, accuracy: 0.6660
Iter : 13001, loss: 1.7035, accuracy: 0.6650
Iter : 14001, loss: 1.7367, accuracy: 0.6750
Iter : 15001, loss: 2.1153, accuracy: 0.6470
Iter : 16001, loss: 1.7145, accuracy: 0.6750
Iter : 17001, loss: 1.9369, accuracy: 0.6510
Iter : 18001, loss: 2.0495, accuracy: 0.6940
Iter : 19001, loss: 1.7785, accuracy: 0.6640
Iter : 20001, loss: 1.9047, accuracy: 0.6470
Iter : 21001, loss: 1.4219, accuracy: 0.6750
Iter : 22001, loss: 1.7642, accuracy: 0.6660
Iter : 23001, loss: